## Spatial preprocessing

### Steps

1. Import Spaceranger outputs  
2. Collect gene and spot-wise QC metrics and write to file 
3. Filter out spots by coordinate   
4. Remove the bottom 1% (by total counts) of spots per sample
5. Remove spots with less than 100 genes
6. Remove spots with >=30% mitochondrial genes per sample
7. Write preprocessed samples individually into `.h5ad` 
8. Concatenate all samples into one
9. Write the concatenated version into `h5ad`

In [1]:
import scanpy as sc
import squidpy as sq
import warnings
import pandas as pd
import numpy as np
from scipy.sparse import block_diag

import matplotlib.pyplot as plt
import seaborn as sns

import os
from sklearn.decomposition import (
    PCA,
)
from sklearn.metrics import adjusted_rand_score 

import scipy.sparse as sp
warnings.filterwarnings("ignore")

/home/ubuntu/miniconda3/envs/spCLUE/lib/python3.9/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/home/ubuntu/miniconda3/envs/spCLUE/lib/python3.9/site-packages/numba/core/decorators.py:246: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


In [2]:
import sys
sys.path.append('/lambda/nfs/jk2/scvi-tools/spclue-code/codes')
import spCLUE
spCLUE.fix_seed(0)

In [3]:
import torch

print(torch.cuda.is_available())  # Should be True
print(torch.version.cuda)         # CUDA version
print(torch.cuda.get_device_name())

True
12.8
NVIDIA GH200 480GB


In [4]:
# Step 1: Read the .txt file as a pandas DataFrame
metadata_df = pd.read_excel("./metadata_batch_effect.xlsx", parse_dates=False)

# Step 2: Initialize an empty list to store AnnData objects
# Another list to store filter stats
filter_df = []

In [5]:
metadata_df.dtypes

library_id                   object
alignment                    object
desired_rc                    int64
actual_rc                     int64
dir                          object
xmin                          int64
xmax                          int64
ymin                          int64
ymax                          int64
hpx                           int64
vpx                           int64
slide                        object
area                         object
date                 datetime64[ns]
cytassist_ver                object
seq_batch                     int64
probe_date           datetime64[ns]
probe_lot                     int64
probe_group                   int64
cond                         object
cond_rep                     object
rep                           int64
age                           int64
sex                          object
bmi                           int64
packyears                   float64
FVC                         float64
DLCO                        

In [6]:
metadata_df['date']= metadata_df['date'].astype('str')

In [7]:
metadata_df['probe_date']= metadata_df['probe_date'].astype('str')

In [8]:
metadata_df.dtypes

library_id            object
alignment             object
desired_rc             int64
actual_rc              int64
dir                   object
xmin                   int64
xmax                   int64
ymin                   int64
ymax                   int64
hpx                    int64
vpx                    int64
slide                 object
area                  object
date                  object
cytassist_ver         object
seq_batch              int64
probe_date            object
probe_lot              int64
probe_group            int64
cond                  object
cond_rep              object
rep                    int64
age                    int64
sex                   object
bmi                    int64
packyears            float64
FVC                  float64
DLCO                 float64
6MWD                 float64
outcome_long          object
tte_long               int64
outcome_24m           object
tte_24m                int64
lobe_lr               object
lobe          

In [9]:
# Step 3: Iterate over each row in the DataFrame to read the Visium datasets
adata_list = []
adata_list_skipone = []
for i in range(len(metadata_df)):
    # Extract the directory path and metadata information
    irow = metadata_df.iloc[i] # get the i-th row
    
    # Step 4: Read the Visium dataset using Scanpy
    adata = sq.read.visium(path=irow['dir'], counts_file = "filtered_feature_bc_matrix.h5", library_id=irow['library_id'])

    # https://stackoverflow.com/questions/74893175/trying-to-concat-a-list-of-12-anndata-objects-but-am-getting-duplicates
    # dont run this because we are using ENSEMBL ID?
    adata.var_names_make_unique() 
    

    # Change gene symbol to ENSEMBL ID
    # adata.var['symbol'] = adata.var_names
    # adata.var.set_index('gene_ids', drop = True, inplace=True)
    
    # Step 5: Add the other metadata columns to the AnnData object
    # You can add any other columns as metadata (e.g., row['other_column_name'])
    for colname in metadata_df.columns:
        if colname not in ['dir','source_image_path']:  # Skip the 'dir' column
            adata.obs[colname] = irow[colname]
    
    # Give each adata a name (sample name)
    adata.name = irow['library_id']


    ## define genes
    # mitochondrial genes, "MT-" for human, "Mt-" for mouse
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    # ribosomal genes
    adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
    # hemoglobin genes
    # .startswith() does not support regex
    adata.var["hb"] = adata.var_names.str.contains("^HB[ABDEGMQZ]")

    # Step 6: Append the AnnData object to the list
    adata_list.append(adata)
    if (adata.name != "20_17478_A1") & (adata.obs['cond'].unique() == "UNC"):
        adata_list_skipone.append(adata)


In [10]:
# Perform quality control metrics calculations and filtering
adata_list_filtered = []
batch_list = []

filter_df = []
for i, adata in enumerate(adata_list_skipone):
      ## This adds additional columns to adata.obs 
    # ex. n_genes_by_genes: number of genes detected for each cell
    # ex. total_genes: total read count for each cell
    # ex. pct for each of mt, ribo and hb
    sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
    )

    ## Filter based on coordinates
    # metadata has four columns (xmin, xmax, ymin and ymax)
    # 0 means no filtering is applied
    # Multiply these values by 25 to get pixel coordinates to filter spots
    adata_name = adata.name

    # Gets the row that mathces the library_id of the adata
    row = metadata_df.loc[metadata_df["library_id"] == adata_name]
    
    #
    if row["xmin"].iloc[0] != 0:
      # adata.obsm["spatial"] is a list of list of x and y coordiantes (i.e. [[X,Y],[X,Y]...])
      # [:,0] means get all [X,Y] lists then pick the 0th index of each, which is X
      # > for greater than "xmin"
      adata = adata[adata.obsm["spatial"][:,0] > 25*adata.obs["xmin"]].copy()

    if row["xmax"].iloc[0] != 0:
      # < for less than "xmax"
      adata = adata[adata.obsm["spatial"][:,0] < 25*adata.obs["xmax"]].copy()

    if row["ymin"].iloc[0] != 0:
      # [:,1] means get all [X,Y] lists then pick the 1th index of each, which is Y
      adata = adata[adata.obsm["spatial"][:,1] > 25*adata.obs["ymin"]].copy()

    if row["ymax"].iloc[0] != 0:
      adata = adata[adata.obsm["spatial"][:,1] < 25*adata.obs["ymax"]].copy()


    ### Obs (spots/cells) filter
    # Store the number of obs before read/gene count filtering
    obs_original = adata.n_obs

    # # adata.X is sparse, so convert to dense
    # data_matrix = adata.X.toarray()
    # # flatten
    # flattened_data = data_matrix.flatten()

    # Calculate various count quantiles using numpy
    # First make a list of quantiles to calculate
    count_quant = [0.01, 0.10, 0.25, 0.5, 0.75, 0.90, 0.99]

    count_quant_np = np.quantile(adata.obs.loc[:,"total_counts"], count_quant)
    

    # Apply filter based on min and max counts
    sc.pp.filter_cells(adata, min_counts=count_quant_np[0], inplace= True)
    #sc.pp.filter_cells(adata, max_counts=count_quant_np[-1], inplace= True)

    # Calculate various gene quantiles using numpy
    gene_quant = [0.10, 0.25, 0.5, 0.75, 0.90]
    gene_quant_np = np.quantile(adata.obs.loc[:,"n_genes_by_counts"], gene_quant)

    sc.pp.filter_cells(adata, min_genes = 100, inplace = True)
    


    



 
   








   

    # Calculate the number of obs removed by mito filter
    obs_removed_mito = adata[adata.obs["pct_counts_mt"]>=30].shape[0]


    # .copy() does not copy the name attribute
    #adata_name = adata.name

    # Only keep obs with <= 30% mito genes
    adata = adata[adata.obs["pct_counts_mt"]<30].copy()


    # Filter out spots whose .obsm['spatial'] arrays contain NaN
    arr1 = ~np.isnan(adata.obsm['spatial'])[:,0] # all lists and their first item (presumably x coordinate)
    arr2 = ~np.isnan(adata.obsm['spatial'])[:,1] # all lists and their second item (presumably y coordinate)
    arr = arr1 & arr2
    adata = adata[arr,:].copy()
    adata.name = adata_name
    adata.layers['counts'] = adata.X.copy()

    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.scale(adata)



    # filter out UNC_6 because it has very few gene counts across spots
    adata_list_filtered.append(adata)
    batch_list += [i] * adata.shape[0]
    
    






In [11]:
batch_list = np.array(batch_list)


In [12]:
np.unique(batch_list)

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])

In [13]:
len(batch_list)

60230

In [14]:
# Concat all adata into one big adata
adata = sc.concat(adata_list_filtered)
# adata = sc.concat(adata_list_filtered, uns_merge="unique", join="outer", 
#                                     index_unique= '-', keys=[adata.name for adata in adata_list_filtered])

In [15]:
adata.obs

,in_tissue,array_row,array_col,library_id,alignment,desired_rc,actual_rc,xmin,xmax,ymin,...,log1p_total_counts_mt,pct_counts_mt,total_counts_ribo,log1p_total_counts_ribo,pct_counts_ribo,total_counts_hb,log1p_total_counts_hb,pct_counts_hb,n_counts,n_genes
AACACTTGGCAAGGAA-1,1.0,47.0,71.0,20_41847_A1,auto,128,151,60,0,0,...,5.710427,2.217801,0.0,0.0,0.0,0.0,0.000000,0.000000,13572.0,5743
AACAGGATTCATAGTT-1,1.0,49.0,43.0,20_41847_A1,auto,128,151,60,0,0,...,6.591674,3.736975,0.0,0.0,0.0,0.0,0.000000,0.000000,19481.0,5684
AACAGGTTATTGCACC-1,1.0,28.0,86.0,20_41847_A1,auto,128,151,60,0,0,...,6.561031,2.959547,0.0,0.0,0.0,0.0,0.000000,0.000000,23855.0,7508
AACAGGTTCACCGAAG-1,1.0,51.0,41.0,20_41847_A1,auto,128,151,60,0,0,...,6.126869,3.367723,0.0,0.0,0.0,0.0,0.000000,0.000000,13570.0,5637
AACAGTCAGGCTCCGC-1,1.0,24.0,6.0,20_41847_A1,auto,128,151,60,0,0,...,6.673298,3.495575,0.0,0.0,0.0,1.0,0.693147,0.004425,22600.0,7606
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TGTTGGAACGAGGTCA-1,1.0,28.0,72.0,12_39986_A2,auto,172,173,0,0,0,...,5.545177,9.914464,0.0,0.0,0.0,0.0,0.000000,0.000000,2572.0,1558
TGTTGGAAGCTCGGTA-1,1.0,1.0,95.0,12_39986_A2,auto,172,173,0,0,0,...,4.564348,3.612167,0.0,0.0,0.0,0.0,0.000000,0.000000,2630.0,1690
TGTTGGATGGACTTCT-1,1.0,13.0,53.0,12_39986_A2,auto,172,173,0,0,0,...,5.442418,7.941989,0.0,0.0,0.0,0.0,0.000000,0.000000,2896.0,1882
TGTTGGCCAGACCTAC-1,1.0,49.0,47.0,12_39986_A2,auto,172,173,0,0,0,...,4.356709,13.873874,0.0,0.0,0.0,0.0,0.000000,0.000000,555.0,423


In [16]:
adata.obsm["X_pca"] = PCA(n_components=200, random_state=0).fit_transform(adata.X)

# spatial graph
g_spatial_list = []
for adata_cur in adata_list_filtered:
    g_spatial = spCLUE.prepare_graph(adata_cur, "spatial")
    g_spatial_list.append(g_spatial)
g_spatial = block_diag(g_spatial_list)

# expression graph
g_expr_list = []
for adata_cur in adata_list_filtered:
    g_expr = spCLUE.prepare_graph(adata_cur, "expr")
    g_expr_list.append(g_expr)
g_expr = block_diag(g_expr_list)

graph_dict = {"spatial": g_spatial, "expr": g_expr}

create adjacent matrix from spatial idx --------------->
create knn graph ---->
spatial knn graph created ----<
create adjacent matrix from spatial idx --------------->
create knn graph ---->
spatial knn graph created ----<
create adjacent matrix from spatial idx --------------->
create knn graph ---->
spatial knn graph created ----<
create adjacent matrix from spatial idx --------------->
create knn graph ---->
spatial knn graph created ----<
create adjacent matrix from spatial idx --------------->
create knn graph ---->
spatial knn graph created ----<
create adjacent matrix from spatial idx --------------->
create knn graph ---->
spatial knn graph created ----<
create adjacent matrix from spatial idx --------------->
create knn graph ---->
spatial knn graph created ----<
create adjacent matrix from spatial idx --------------->
create knn graph ---->
spatial knn graph created ----<
create adjacent matrix from spatial idx --------------->
create knn graph ---->
spatial knn graph create

In [17]:
adata.obsm['X_pca']

array([[ 1.2249450e+01, -8.8971052e+00, -4.6114850e+00, ...,
        -8.6096758e-01, -2.1027524e-02,  6.5454680e-01],
       [ 4.3449650e+00, -8.1793470e+00, -2.8632982e+01, ...,
         4.0174642e-01,  1.0847760e+00,  7.7843839e-01],
       [ 1.3667771e+01, -4.8771834e+00,  6.7720985e+00, ...,
         1.8415953e+00,  4.1087118e-01, -5.4904491e-01],
       ...,
       [ 1.1255548e+01, -6.6324649e+00,  6.7767873e+00, ...,
         1.1094810e+00,  6.2225927e-02,  1.1958023e-01],
       [-1.5295207e+01,  7.1406355e+00,  8.2924265e-01, ...,
         9.7059160e-01,  1.7258013e+00, -4.5875084e-01],
       [ 1.2207072e+01, -1.9161531e+00,  1.2341843e+00, ...,
         2.8071078e-02,  2.0796251e+00, -8.4118247e-02]], dtype=float32)

In [18]:
batch_list

array([ 0,  0,  0, ..., 15, 15, 15])

In [19]:
len(batch_list)

60230

In [20]:
np.unique(batch_list)

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])

In [21]:
import torch
for n in range(4,20):
    spCLUE_model = spCLUE.spCLUE(adata.obsm["X_pca"], graph_dict, n, batch_list, device="cuda:0", batch_train=True)
    _, adata.obsm["spCLUE"] = spCLUE_model.trainBatch()
    adata.obs["batch_name"] = batch_list
    adata.obs["batch_name"] = adata.obs["batch_name"].astype("category")
    cluster_method = "mclust"
    
    spCLUE.clustering(adata,
                             n,
                             key='spCLUE',
                             cluster_methods=cluster_method)
    spCLUE.batch_refine_label(adata, key="mclust", batch_key="batch_name")
    adata.write(f'/home/ubuntu/jk2/scvi-tools/h5ad/adata_filtered_concat_only_UNC_spCLUE_n_clusters_{n}.h5ad')

Training Start =========================>


 21%|██        | 105/500 [00:04<00:12, 32.90it/s]

epoch 100: 0.26


 41%|████      | 205/500 [00:07<00:08, 33.29it/s]

epoch 200: 0.33


...

epoch 300: 0.16


 81%|████████  | 404/500 [00:12<00:03, 30.74it/s]

epoch 400: 0.16


100%|██████████| 500/500 [00:15<00:00, 31.36it/s]


epoch 500: 0.16
Training Finished =================<
fitting ...
  |======================================================================| 100%
